# 02 · Merge samples and plot QC metrics

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`merge_and_plot_qc_metrics`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`merge_and_plot_qc`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/merge_and_plot_qc) → `sc-merge-and-plot-qc` |

`merge_and_plot_qc` concatenates the per-sample AnnData objects into a single cohort object:

- Samples are read from a two-column TSV (*sample name*, *path*). The WDL builds it by stripping `.cleaned_unfiltered.h5ad` from each file name, so the sample name is `${dataset_id}.${sample_id}`.
- Samples are concatenated in chunks of 50 with `anndata.concat(merge="same", uns_merge="same", index_unique="_")`. Barcodes repeat across samples, so each cell is renamed `<barcode>_<sample name>`. `merge="same"` keeps only `var` columns that agree in every sample.
- Violin plots of `n_genes_by_counts`, `total_counts`, `pct_counts_mt`, `pct_counts_rb` and `doublet_score` show the cells before any filtering, which helps pick thresholds for the next step.

The workflow can also downsample to `n_cells` here; that is not used (all cells are kept).

**Inputs**
- `*.cleaned_unfiltered.h5ad` for every sample (via a file-of-filenames TSV)

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.merged_cleaned_unfiltered.h5ad`
- `${cohort_id}.initial_metadata.csv`
- `plots/${cohort_id}.<metric>.violin.png` for 5 QC metrics

**Previous:** [01_prep_metadata](01_prep_metadata.ipynb) · **Next:** [03_filter](03_filter.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Write the sample file-of-filenames

This is the shell loop at the top of the WDL task's `command` block.

In [ ]:
fofn = cfg.RUN_DIR / "adata_samples_paths.tsv"
cleaned = sorted(cfg.PREPROCESS_PATH.glob("*.cleaned_unfiltered.h5ad"))
with open(fofn, "w") as f:
    for path in cleaned:
        f.write(f"{path.name.removesuffix('.cleaned_unfiltered.h5ad')}\t{path.resolve()}\n")
print(fofn.read_text())

## 2. Run `sc-merge-and-plot-qc`

In [ ]:
cfg.run(
    "sc-merge-and-plot-qc",
    "--adata-objects-fofn", fofn,
    "--adata-output", cfg.out("merged_cleaned_unfiltered.h5ad"),
    "--output-metadata-file", cfg.out("initial_metadata.csv"),
)

The WDL renames the plots to `${cohort_id}.<metric>.violin.png`; do the same, then show them.

In [ ]:
qc_metrics = ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_rb", "doublet_score"]
plots = cfg.RUN_DIR / "plots"
for m in qc_metrics:
    src = plots / f"violin_{m}.png"
    dst = plots / f"{cfg.COHORT_ID}.{m}.violin.png"
    if src.exists():
        src.replace(dst)
    display(Image(filename=str(dst), width=700))

## 3. Inspect the merged metadata

In [ ]:
initial = pd.read_csv(cfg.out("initial_metadata.csv"), index_col=0)
print(f"{len(initial):,} cells, {initial['sample'].nunique()} samples, {initial['batch_id'].nunique()} batches")
initial[qc_metrics].describe().T

## 4. Compare with the curated release

In [ ]:
curated_initial = cfg.curated("initial_metadata.csv")
if curated_initial.exists():
    released = pd.read_csv(curated_initial, index_col=0, usecols=lambda c: c in {"Unnamed: 0", "sample"})
    comparison = pd.DataFrame({
        "re-run": initial["sample"].value_counts(),
        "release": released["sample"].value_counts(),
    })
    comparison["difference"] = comparison["re-run"] - comparison["release"]
    print(f"cells: re-run {len(initial):,}, release {len(released):,}")
    display(comparison)
else:
    print(f"{curated_initial.name} not in the curated release")

**Previous:** [01_prep_metadata](01_prep_metadata.ipynb) · **Next:** [03_filter](03_filter.ipynb)